In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider="openai",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY"),
)

agent = create_agent(model=model, tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response)

{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='f0fbba64-be87-4957-a8e0-a1df003e003e'), HumanMessage(content='你好，我是虎哥.', additional_kwargs={}, response_metadata={}, id='a5204adb-2c29-432d-a87f-a6ce9d153849'), AIMessage(content='你好，虎哥，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='b4224428-f99b-4028-8d01-be5e3497d5ea', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='6efae9d3-9e58-43a6-b4e8-abbcfd87f2ec'), AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 318, 'total_tokens': 382, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.5-4B', 'system_fingerprint': '', 'id': 'chatcmpl-8db49e039581b7e3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1ef8-00e9-7660-a9c1-c1a925c48ef1-0

In [3]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_465b96c6f18f4a218d2e8d46)
 Call ID: call_465b96c6f18f4a218d2e8d46
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================



虎哥，今天北京的天气是晴朗的（sunny）。祝您心情长虹！


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- .env 中配置的多模态模型
- gpt-5-nano
- ...

我们以 .env 中配置的多模态模型为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [4]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),  # 模型名称，从 .env 中读取
    model_provider="openai",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY")
)

In [5]:
# 创建Agent
agent = create_agent(model=model)

In [6]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [7]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)



这张照片捕捉了一个温馨、愉快的瞬间，展示了一位年轻女子和她的宠物狗在沙滩上的亲密互动。以下是画面的详细描述：

**1. 主要人物与动物：**
*   **狗狗：** 画面左侧坐着一只浅棕色的拉布拉多犬。它看起来非常温顺，身上佩戴着带有彩色图案（似乎是脚印形状）的胸背带，并系着红色的牵引绳。
*   **人物：** 画面右侧坐着一位年轻女子，她留着深色的长发，身穿黑白格子的长袖衬衫和深色长裤。她的脸上洋溢着灿烂的笑容，眼神温柔地看着狗狗。

**2. 动作与互动：**
*   狗狗正抬起它的前爪，与女子的左手轻轻触碰，看起来像是在正在进行“握手”という宠物训练，或者是在亲切地互相打招呼。女子的左手正好放在狗狗的爪子上，配合着训练。

**3. 环境与背景：**
*   场景位于一片开阔的海滩上，脚下的沙子细腻柔软，可以看到一些自然的脚印和纹路。
*   背景是平静的海面，远处有一道白色的浪花正向岸边涌来。海平面与天空相接，视野开阔。

**4. 光线与氛围：**
*   从画面的光线判断，时间应该是在日落时分（Golden Hour）。夕阳的余晖从右上方洒下，给女子的头发和狗狗的背部镀上了一层柔和的金边（轮廓光）。
*   整体色调温暖、明亮，营造出一种宁静、和谐且充满幸福感的氛围。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [12]:
!uv add ipywidgets

Resolved 198 packages in 1ms
Checked 192 packages in 6ms


然后，我们创建一个上传组件，用于模拟图片上传。


In [9]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [14]:
print(uploader.value)

({'name': 'foodimg.png', 'type': 'image/png', 'size': 1512560, 'content': <memory at 0x00000136FF986E00>, 'last_modified': datetime.datetime(2026, 5, 12, 8, 54, 17, 717000, tzinfo=datetime.timezone.utc)},)


In [15]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)